## Logistic Regression Model

In [60]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
import seaborn as sns
import re
import nltk
nltk.download('stopwords')



[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/liameikill/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [46]:
df = pd.read_csv("reviews.csv", encoding="utf-8")
df.head()

,review_id,rating,review_text,review_date,helpful
0,981e465b-d3ba-4632-9c60-25051efac38a,5,It's good,11/22/2025 1:19,0
1,964d3555-9429-4c20-8127-ce3c71ce9273,5,WhatsApp not working well always shows offline...,11/24/2025 20:03,0
2,6c28859f-1554-4ca1-9aa8-9d66f204be0a,5,"Oppo not corresponding, share with me the offi...",11/25/2025 6:26,0
3,a7efafc3-5871-4020-a398-9cc12cb4072a,5,"Excellent app, great communication super conne...",11/25/2025 18:09,0
4,de142b31-a5ad-446f-b7c8-51b264728478,4,simply the ɓest for chat and calls.i love it,11/24/2025 1:10,1


## Text Preprocessing

- **Lowercasing:** Reduces vocabulary size 
- **Remove URLs / emails / special characters:** These carry no sentiment and just add noise
- **Remove extra whitespace:** Normalises the input
- **Stop words:** We intentionally keep some stop words like "not", "no", "very" because they carry sentiment.

In [61]:
df = pd.read_csv("reviews.csv")
print(f"Shape: {df.shape}")


stop_words = set(stopwords.words("english"))

sentiment_keepers = {
    'not','no','nor','never','very','too','most','more','less','least',
    'few','but','however','although','only','just'
}

stop_words = stop_words - sentiment_keepers

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text) #URL 
    text = re.sub(r"\S+@\S+", "", text) #Email
    text = re.sub(r"[^a-z\s]", "", text) #Non-letters
    text = re.sub(r"\s+", " ", text).strip()
    words = [w for w in text.split() if w not in stop_words]

    return " ".join(words)

df["clean_text"] = df["review_text"].apply(clean_text)

# Remove empty rows
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

print("\nRemaining rows:", df.shape[0])

Shape: (6210, 5)

Remaining rows: 6168


## TF-IDF Feature Extraction

Logistic regression models cannot process raw text directly, since they require numerical input. Therefore, the cleaned review text must be converted into a numerical representation that the model can use. One common technique for this in natural language processing is Term Frequency–Inverse Document Frequency (TF-IDF).


TF-IDF represents each document (in this case, each review) as a vector of numbers that reflect how important each word is within the dataset. It works by combining two components:

- **Term Frequency (TF)**: Measures how frequently a word appears in a specific document.
- **Inverse Document Frequency (IDF)**: Reduces the importance of words that appear very frequently across many documents (such as common words like “the” or “app”), while increasing the importance of words that are more unique to specific reviews.


By combining these two measures, TF-IDF highlights words that are particularly informative for distinguishing between different reviews.
In this project, TF-IDF is applied to the cleaned review text to transform each review into a numerical feature vector. 

These vectors are then used as input to the logistic regression model.
The vectorizer is configured with the following parameters:

- **max_features=5000**: Limits the vocabulary to the 5000 most informative words to reduce dimensionality and improve computational efficiency.
- **ngram_range=(1,2)**: Includes both single words (unigrams) and pairs of words (bigrams), allowing the model to capture short phrases such as “not working” or “very good”.
- **min_df=5**: Ignores words that appear in fewer than five reviews, helping to remove noise and extremely rare terms.


In [62]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),  
    min_df=5
)

X = vectorizer.fit_transform(df["clean_text"])
y = df["rating"]